In [0]:
print(
    "Final Gold Records :",
    spark.table("gold_daily_revenue").count()
)

Final Gold Records : 60


In [0]:
display(spark.table("gold_daily_revenue"))

txn_date,daily_revenue
2024-01-19,54107
2024-02-20,50077
2024-01-22,71970
2024-01-10,44194
2024-02-03,51253
2024-01-11,59362
2024-01-06,61466
2024-02-04,57314
2024-02-27,53949
2024-02-10,48683


In [0]:
display(spark.table("gold_daily_revenue"))

txn_date,daily_revenue
2024-01-19,54107
2024-02-20,50077
2024-01-22,71970
2024-01-10,44194
2024-02-03,51253
2024-01-11,59362
2024-01-06,61466
2024-02-04,57314
2024-02-27,53949
2024-02-10,48683


In [0]:
from delta.tables import DeltaTable

gold_table = DeltaTable.forName(
    spark,
    "gold_daily_revenue"
)

(
    gold_table.alias("old")
    .merge(
        late_revenue_df.alias("new"),
        "old.txn_date = new.txn_date"
    )
    .whenMatchedUpdate(
        set={
            "daily_revenue": "new.daily_revenue"
        }
    )
    .whenNotMatchedInsert(
        values={
            "txn_date": "new.txn_date",
            "daily_revenue": "new.daily_revenue"
        }
    )
    .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
late_revenue_df.createOrReplaceTempView("late_revenue_updates")

In [0]:
display(late_revenue_df)

txn_date,daily_revenue
2024-01-07,55980
2024-02-26,60249
2024-01-09,79185
2024-01-17,40388
2024-01-06,61466
2024-01-08,82189
2024-02-18,36045
2024-02-12,59226
2024-02-09,64503
2024-01-16,66834


In [0]:
from pyspark.sql.functions import sum

late_revenue_df = (
    late_transactions
    .groupBy("txn_date")
    .agg(
        sum("amount").alias("daily_revenue")
    )
)

In [0]:
print(
    "Affected Dates :",
    late_transactions.select("txn_date").distinct().count()
)

Affected Dates : 60


In [0]:
display(
    late_transactions
    .select("txn_date")
    .distinct()
)

txn_date
2024-01-07
2024-02-25
2024-02-26
2024-01-04
2024-01-09
2024-01-21
2024-01-17
2024-01-31
2024-01-06
2024-01-08


In [0]:
display(late_transactions)

txn_id,user_id,txn_date,amount,ingestion_date,_rescued_data
12,294,2024-01-07,2917,2024-01-10,null
38,449,2024-02-26,2285,2024-03-07,null
70,329,2024-01-09,1603,2024-01-14,null
161,330,2024-01-17,2371,2024-01-18,null
190,222,2024-01-09,1037,2024-01-16,null
218,108,2024-01-06,1915,2024-01-08,null
225,261,2024-01-08,1616,2024-01-18,null
257,324,2024-01-08,2591,2024-01-15,null
263,110,2024-02-18,4820,2024-02-20,null
273,100,2024-02-12,3884,2024-02-22,null


In [0]:
print("Late Transactions :", late_transactions.count())

Late Transactions : 1415


In [0]:
late_transactions = (
    late_df
    .filter(col("txn_date") < col("ingestion_date"))
)

In [0]:
late_df = spark.table("silver_transactions")

In [0]:
from pyspark.sql.functions import col